# 19 · Matched train/validation soft Dice in inference mode (no training)
**Runtime: T4 GPU** (about 30–45 min for 190 runs; CPU works but is slow). Read-only for all run folders; writes `results/p2_jit_gap/`.

This notebook gives a train–validation comparison measured under the **same** conditions. For every finished run, the saved best
checkpoint is applied in inference mode (dropout off, batch-norm moving statistics, **no augmentation**) to the run's own training
crops (3 folds, 240 ROIs) and validation crops (1 fold, 80 ROIs). We record the mean per-image soft Dice (smoothing 1, as in the loss)
and the thresholded Dice on both sets. As a check, the inference-mode validation soft Dice is compared with the value logged in
`history.csv` at the best epoch. Resumable: one row per run is appended; finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
SETS     = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
OUT      = f'{PROJECT}/results/p2_jit_gap'; os.makedirs(OUT, exist_ok=True)
CODE_DIR = f'{PROJECT}/code'
QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
from qseg.config import Config
cfg = Config(work_dir=OUT)
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'), '| CBIS', XC.shape)

In [ ]:
def soft_dice(Y, P):
    y = Y.reshape(len(Y), -1); p = P.reshape(len(P), -1)
    return (2 * (y * p).sum(1) + 1) / (y.sum(1) + p.sum(1) + 1)
def hard_dice(Y, P, thr):
    y = Y.reshape(len(Y), -1) > 0.5; p = P.reshape(len(P), -1) >= thr
    s = y.sum(1) + p.sum(1); return np.where(s == 0, 1.0, 2 * (y & p).sum(1) / np.maximum(s, 1))
DST = f'{OUT}/gap_runs.csv'
done = set(pd.read_csv(DST).run_key) if os.path.exists(DST) else set()
n = 0
for set_name, root in SETS.items():
    runs = [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
    print(set_name, len(runs), 'finished runs')
    for d in runs:
        key = f'{set_name}/{os.path.basename(d)}'
        if key in done: continue
        t0 = time.time(); s = json.load(open(f'{d}/summary.json'))
        name, fold, seed = s['model'], int(s['fold']), int(s['seed'])
        tf.keras.backend.clear_session()
        m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
        m.load_weights(f'{d}/best.weights.h5')
        tr, va, te = qseg2.fold_split(DFC, fold)
        Ptr = m.predict(XC[tr], batch_size=cfg.batch_size, verbose=0); Pva = m.predict(XC[va], batch_size=cfg.batch_size, verbose=0)
        row = dict(run_key=key, set=set_name, model=name, fold=fold, seed=seed, n_train=int(len(tr)), n_val=int(len(va)),
                   train_soft_dice_infer=float(soft_dice(YC[tr], Ptr).mean()), val_soft_dice_infer=float(soft_dice(YC[va], Pva).mean()),
                   train_dice_infer=float(hard_dice(YC[tr], Ptr, cfg.threshold).mean()), val_dice_infer=float(hard_dice(YC[va], Pva, cfg.threshold).mean()),
                   best_epoch=s.get('best_epoch'))
        h = f'{d}/history.csv'
        if os.path.exists(h):
            H = pd.read_csv(h); b = H.loc[[H.val_loss.idxmin()]]          # best checkpoint = lowest validation loss
            if len(b): row.update(train_soft_dice_logged=float(b.soft_dice.iloc[0]), val_soft_dice_logged=float(b.val_soft_dice.iloc[0]))
        row['gap_matched'] = row['train_soft_dice_infer'] - row['val_soft_dice_infer']
        pd.DataFrame([row]).to_csv(DST, mode='a', header=not os.path.exists(DST), index=False); n += 1
        print(f"{key:34s} train {row['train_soft_dice_infer']:.4f} val {row['val_soft_dice_infer']:.4f} gap {row['gap_matched']:+.4f} "
              f"| logged val {row.get('val_soft_dice_logged', float('nan')):.4f} ({time.time() - t0:.0f} s)", flush=True)
print('new runs:', n)

In [ ]:
G = pd.read_csv(DST)
G['val_check'] = G.val_soft_dice_infer - G.val_soft_dice_logged
print('runs:', len(G), '| |inference val - logged val| max', round(G.val_check.abs().max(), 4), 'median', round(G.val_check.abs().median(), 5))
tab = G.groupby(['set', 'model']).agg(n=('gap_matched', 'size'), gap_mean=('gap_matched', 'mean'), gap_min=('gap_matched', 'min'), gap_max=('gap_matched', 'max'),
                                     train_soft=('train_soft_dice_infer', 'mean'), val_soft=('val_soft_dice_infer', 'mean')).round(4)
print(tab.to_string()); tab.to_csv(f'{OUT}/gap_summary.csv')

In [ ]:
# Auto-shutdown: save Drive, then release the GPU
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()